<a href="https://colab.research.google.com/github/kvssri/online-tihiitg/blob/main/Project-Specific%20Robustness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

print("Uploaded file:", list(uploaded.keys())[0])

Saving Underwater-Image-Data-set-main.zip to Underwater-Image-Data-set-main.zip
Uploaded file: Underwater-Image-Data-set-main.zip


In [2]:
import zipfile
import os
import glob
import pandas as pd
import numpy as np

zip_file = list(uploaded.keys())[0]

extract_path = "/content/robustness_data"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

csv_files = glob.glob(
    extract_path + "/**/*.csv",
    recursive=True
)

print("CSV files found:", len(csv_files))

for f in csv_files:
    print(f)

CSV files found: 8
/content/robustness_data/Underwater-Image-Data-set-main/training_log 2.csv
/content/robustness_data/Underwater-Image-Data-set-main/training_log 4.csv
/content/robustness_data/Underwater-Image-Data-set-main/training_log 6.csv
/content/robustness_data/Underwater-Image-Data-set-main/training_log 8.csv
/content/robustness_data/Underwater-Image-Data-set-main/training_log 1.csv
/content/robustness_data/Underwater-Image-Data-set-main/training_log 5.csv
/content/robustness_data/Underwater-Image-Data-set-main/training_log 7.csv
/content/robustness_data/Underwater-Image-Data-set-main/training_log 3.csv


In [3]:
all_logs = []

for f in csv_files:
    temp = pd.read_csv(f)

    # Keep the original training-run identity
    temp["source_file"] = os.path.basename(f)

    all_logs.append(temp)

df = pd.concat(all_logs, ignore_index=True)

print("Combined dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nSamples per source:")
print(df["source_file"].value_counts())


Combined dataset shape: (24915, 6)

Columns:
['epoch', 'step', 'gen_total', 'disc_loss', 'time_s', 'source_file']

Samples per source:
source_file
training_log 4.csv    3900
training_log 7.csv    3900
training_log 5.csv    3900
training_log 1.csv    3900
training_log 3.csv    3900
training_log 2.csv    3660
training_log 6.csv    1521
training_log 8.csv     234
Name: count, dtype: int64


In [4]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nBasic statistics:")
display(df[["epoch", "step", "gen_total", "disc_loss", "time_s"]].describe())

Missing values:
epoch          0
step           0
gen_total      0
disc_loss      0
time_s         0
source_file    0
dtype: int64

Duplicate rows: 0

Basic statistics:


,epoch,step,gen_total,disc_loss,time_s
count,24915.000000,24915.000000,24915.000000,24915.000000,24915.000000
mean,52.333935,1835.712462,13.347087,0.551729,96.163017
std,29.743196,1129.729162,7.621047,0.292557,213.210639
min,1.000000,1.000000,3.235735,0.020976,4.876095
25%,27.000000,850.000000,7.840670,0.418167,45.790789
50%,53.000000,1776.000000,11.780836,0.548061,79.022856
75%,77.000000,2814.000000,16.629889,0.655409,112.391965
max,136.000000,3900.000000,90.369987,11.538145,7865.182561


In [5]:
# Sort data chronologically
df = df.sort_values(
    ["source_file", "epoch", "step"]
).reset_index(drop=True)

# Use the first 70% of records to determine thresholds
train_cutoff = int(0.70 * len(df))
train_gen = df.loc[:train_cutoff-1, "gen_total"]

low_threshold = train_gen.quantile(0.33)
high_threshold = train_gen.quantile(0.67)

def assign_condition(value):
    if value <= low_threshold:
        return "Low"
    elif value <= high_threshold:
        return "Medium"
    else:
        return "High"

df["condition"] = df["gen_total"].apply(assign_condition)

print("Condition thresholds:")
print("Low threshold:", low_threshold)
print("High threshold:", high_threshold)

print("\nCondition distribution:")
print(df["condition"].value_counts())

Condition thresholds:
Low threshold: 9.512517356872559
High threshold: 14.876732540130616

Condition distribution:
condition
Low       8966
Medium    7995
High      7954
Name: count, dtype: int64


In [6]:
features = ["epoch", "step", "disc_loss", "time_s"]
target = "gen_total"

X = df[features].values
y = df[target].values

SEQUENCE_LENGTH = 10

X_seq = []
y_seq = []
source_seq = []
condition_seq = []

for i in range(len(X) - SEQUENCE_LENGTH):
    X_seq.append(X[i:i + SEQUENCE_LENGTH])
    y_seq.append(y[i + SEQUENCE_LENGTH])

    # Condition/source of the prediction target
    source_seq.append(df.iloc[i + SEQUENCE_LENGTH]["source_file"])
    condition_seq.append(df.iloc[i + SEQUENCE_LENGTH]["condition"])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)
source_seq = np.array(source_seq)
condition_seq = np.array(condition_seq)

print("Sequence input shape:", X_seq.shape)
print("Sequence target shape:", y_seq.shape)
print("Source labels:", len(source_seq))
print("Condition labels:", len(condition_seq))

Sequence input shape: (24905, 10, 4)
Sequence target shape: (24905,)
Source labels: 24905
Condition labels: 24905


In [7]:
n = len(X_seq)

train_end = int(0.70 * n)
val_end = int(0.85 * n)

X_train = X_seq[:train_end]
y_train = y_seq[:train_end]

X_val = X_seq[train_end:val_end]
y_val = y_seq[train_end:val_end]

X_test = X_seq[val_end:]
y_test = y_seq[val_end:]

source_train = source_seq[:train_end]
source_val = source_seq[train_end:val_end]
source_test = source_seq[val_end:]

condition_train = condition_seq[:train_end]
condition_val = condition_seq[train_end:val_end]
condition_test = condition_seq[val_end:]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTest conditions:")
print(pd.Series(condition_test).value_counts())

Train: (17433, 10, 4)
Validation: (3736, 10, 4)
Test: (3736, 10, 4)

Test conditions:
High      1703
Medium    1502
Low        531
Name: count, dtype: int64


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
scaler.fit(X_train_2d)

# Transform train, validation and test using the training scaler
X_train = scaler.transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_val = scaler.transform(
    X_val.reshape(-1, X_val.shape[-1])
).reshape(X_val.shape)

X_test = scaler.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

print("Training-only normalization completed.")
print("Train mean:", X_train.mean(axis=(0, 1)))
print("Train std:", X_train.std(axis=(0, 1)))

Training-only normalization completed.
Train mean: [-1.78447797e-15 -5.00503098e-16 -6.90367284e-16 -4.56635381e-16]
Train std: [1. 1. 1. 1.]


In [9]:
import torch
import torch.nn as nn

class TCNModel(nn.Module):
    def __init__(self, input_size, hidden_channels=64, dropout=0.0):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(
                input_size,
                hidden_channels,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                hidden_channels,
                hidden_channels,
                kernel_size=3,
                padding=2,
                dilation=2
            ),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                hidden_channels,
                32,
                kernel_size=3,
                padding=4,
                dilation=4
            ),
            nn.ReLU()
        )

        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.network(x)
        x = x[:, :, -1]
        return self.fc(x)

In [10]:
import json
import os

config_path = "/content/best_tuned_configuration.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        best_config = json.load(f)

    print("Best configuration loaded:")
    print(json.dumps(best_config, indent=4))
else:
    print("Configuration file not found.")
    print("Please upload best_tuned_configuration.json")

Configuration file not found.
Please upload best_tuned_configuration.json


In [12]:
import json
import os

config_path = "/content/best_tuned_configuration.json"

if not os.path.exists(config_path):
    print("Please upload best_tuned_configuration.json")
    from google.colab import files
    uploaded_config = files.upload()
    config_path = "/content/" + list(uploaded_config.keys())[0]

with open(config_path, "r") as f:
    best_config = json.load(f)

print("Best configuration loaded successfully:")
print(json.dumps(best_config, indent=4))

Please upload best_tuned_configuration.json


Saving _Parameter Experiments.ipynb to _Parameter Experiments.ipynb
Best configuration loaded successfully:
{
    "nbformat": 4,
    "nbformat_minor": 0,
    "metadata": {
        "colab": {
            "provenance": [],
            "authorship_tag": "ABX9TyO1Wy/8waYDUeDuBGeSQQRv",
            "include_colab_link": true
        },
        "kernelspec": {
            "name": "python3",
            "display_name": "Python 3"
        },
        "language_info": {
            "name": "python"
        }
    },
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {
                "id": "view-in-github",
                "colab_type": "text"
            },
            "source": [
                "<a href=\"https://colab.research.google.com/github/kvssri/online-tihiitg/blob/main/Controlled%20Hyperparameter%20/%20Parameter%20Experiments.ipynb\" target=\"_parent\"><img src=\"https://colab.research.google.com/assets/colab-badge.svg\" alt=\"Open In Colab\"/></a>"
 

In [14]:
print("Configuration contents:")
print(best_config)

print("\nAvailable keys:")
print(list(best_config.keys()))

Configuration contents:
{'nbformat': 4, 'nbformat_minor': 0, 'metadata': {'colab': {'provenance': [], 'authorship_tag': 'ABX9TyO1Wy/8waYDUeDuBGeSQQRv', 'include_colab_link': True}, 'kernelspec': {'name': 'python3', 'display_name': 'Python 3'}, 'language_info': {'name': 'python'}}, 'cells': [{'cell_type': 'markdown', 'metadata': {'id': 'view-in-github', 'colab_type': 'text'}, 'source': ['<a href="https://colab.research.google.com/github/kvssri/online-tihiitg/blob/main/Controlled%20Hyperparameter%20/%20Parameter%20Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>']}, {'cell_type': 'code', 'source': ['from google.colab import files\n', '\n', 'uploaded = files.upload()'], 'metadata': {'colab': {'base_uri': 'https://localhost:8080/', 'height': 73}, 'id': 'fZPEnzrmq9XU', 'outputId': '48c19ffa-01aa-4ec5-e250-c408f36912a5'}, 'execution_count': 2, 'outputs': [{'output_type': 'display_data', 'data': {'text/plain': ['

In [15]:
# Best configuration from Day 19: TCN-4

hidden_channels = 64
dropout = 0.0

model = TCNModel(
    input_size=X_train.shape[2],
    hidden_channels=hidden_channels,
    dropout=dropout
)

print("Tuned TCN created successfully.")
print("Configuration:")
print("Hidden channels:", hidden_channels)
print("Dropout:", dropout)
print("Learning rate:", 0.0005)
print("Batch size:", 32)

Tuned TCN created successfully.
Configuration:
Hidden channels: 64
Dropout: 0.0
Learning rate: 0.0005
Batch size: 32


In [18]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import time

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Best Day 19 configuration
hidden_channels = 64
dropout = 0.0
learning_rate = 0.0005
batch_size = 32

# Create model
model = TCNModel(
    input_size=X_train.shape[2],
    hidden_channels=hidden_channels,
    dropout=dropout
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

# Data loaders
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    TensorDataset(X_val_tensor, y_val_tensor),
    batch_size=batch_size,
    shuffle=False
)

# Training
epochs = 15
best_val_loss = float("inf")

train_losses = []
val_losses = []

start_time = time.time()

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        predictions = model(xb)
        loss = criterion(predictions, yb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            predictions = model(xb)
            loss = criterion(predictions, yb)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

runtime = time.time() - start_time

print("\nTraining completed!")
print("Best validation loss:", best_val_loss)
print("Runtime:", runtime)

Epoch 1/15 | Train Loss: 75.3528 | Val Loss: 22.4314
Epoch 2/15 | Train Loss: 41.0394 | Val Loss: 20.6775
Epoch 3/15 | Train Loss: 37.9452 | Val Loss: 20.2778
Epoch 4/15 | Train Loss: 36.1191 | Val Loss: 20.1149
Epoch 5/15 | Train Loss: 34.8981 | Val Loss: 19.9434
Epoch 6/15 | Train Loss: 34.1490 | Val Loss: 19.8669
Epoch 7/15 | Train Loss: 33.5756 | Val Loss: 19.6977
Epoch 8/15 | Train Loss: 33.0235 | Val Loss: 19.6616
Epoch 9/15 | Train Loss: 32.7483 | Val Loss: 19.7339
Epoch 10/15 | Train Loss: 32.5774 | Val Loss: 19.3763
Epoch 11/15 | Train Loss: 32.2532 | Val Loss: 19.2647
Epoch 12/15 | Train Loss: 32.0671 | Val Loss: 19.1448
Epoch 13/15 | Train Loss: 31.9504 | Val Loss: 18.9833
Epoch 14/15 | Train Loss: 31.8271 | Val Loss: 18.7899
Epoch 15/15 | Train Loss: 31.7074 | Val Loss: 18.6425

Training completed!
Best validation loss: 18.642462025340805
Runtime: 46.58477711677551


In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor).numpy().flatten()

y_true = y_test_tensor.numpy().flatten()

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# Safe MAPE
mape = np.mean(
    np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))
) * 100

r2 = r2_score(y_true, y_pred)

print("TCN Test Results")
print("---------------------------")
print("MAE  :", mae)
print("RMSE :", rmse)
print("MAPE :", mape, "%")
print("R²   :", r2)

TCN Test Results
---------------------------
MAE  : 5.843605041503906
RMSE : 7.430473021368765
MAPE : 35.272415 %
R²   : -1.1771795749664307


In [20]:
import pandas as pd
import numpy as np

results = []

for condition in np.unique(condition_test):

    mask = condition_test == condition

    actual = y_true[mask]
    predicted = y_pred[mask]

    condition_mae = np.mean(np.abs(actual - predicted))
    condition_rmse = np.sqrt(np.mean((actual - predicted) ** 2))

    condition_mape = np.mean(
        np.abs((actual - predicted) /
               np.maximum(np.abs(actual), 1e-8))
    ) * 100

    results.append({
        "Condition": condition,
        "Samples": len(actual),
        "MAE": condition_mae,
        "RMSE": condition_rmse,
        "MAPE (%)": condition_mape
    })

robustness_table = pd.DataFrame(results)

print("Robustness Analysis by Condition")
display(robustness_table)

Robustness Analysis by Condition


,Condition,Samples,MAE,RMSE,MAPE (%)
0,High,1703,9.417914,10.351104,47.524002
1,Low,531,1.361175,1.782433,20.371580
2,Medium,1502,3.375643,3.837253,26.649166


In [21]:
results_source = []

for source in np.unique(source_test):

    mask = source_test == source

    actual = y_true[mask]
    predicted = y_pred[mask]

    source_mae = np.mean(np.abs(actual - predicted))
    source_rmse = np.sqrt(np.mean((actual - predicted) ** 2))

    source_mape = np.mean(
        np.abs((actual - predicted) /
               np.maximum(np.abs(actual), 1e-8))
    ) * 100

    results_source.append({
        "Source/Run": source,
        "Samples": len(actual),
        "MAE": source_mae,
        "RMSE": source_rmse,
        "MAPE (%)": source_mape
    })

cross_run_table = pd.DataFrame(results_source)

print("Cross-Run Robustness Analysis")
display(cross_run_table)

Cross-Run Robustness Analysis


,Source/Run,Samples,MAE,RMSE,MAPE (%)
0,training_log 7.csv,3502,5.711489,7.295892,34.390030
1,training_log 8.csv,234,7.820847,9.212634,48.477924


In [22]:
# Combine condition and cross-run results
robustness_table.to_csv(
    "/content/condition_robustness_results.csv",
    index=False
)

cross_run_table.to_csv(
    "/content/cross_run_robustness_results.csv",
    index=False
)

# Save overall test metrics
overall_results = pd.DataFrame([{
    "Method": "Tuned TCN",
    "MAE": mae,
    "RMSE": rmse,
    "MAPE (%)": mape,
    "R2": r2
}])

overall_results.to_csv(
    "/content/overall_tcn_results.csv",
    index=False
)

print("All robustness results saved successfully.")
print("\nFiles created:")
print("1. condition_robustness_results.csv")
print("2. cross_run_robustness_results.csv")
print("3. overall_tcn_results.csv")

All robustness results saved successfully.

Files created:
1. condition_robustness_results.csv
2. cross_run_robustness_results.csv
3. overall_tcn_results.csv


## Research Insight

The robustness analysis evaluated the tuned TCN model across different
source/training runs and target-behaviour conditions.

The results show how prediction error changes under different operating
conditions. Lower MAE and RMSE indicate more stable prediction performance,
while higher errors indicate conditions where the model has greater difficulty.

The cross-run analysis helps identify whether the model performance is
consistent across different training sources. The condition-wise analysis
shows whether prediction accuracy changes for different target-behaviour
ranges.

### Research Question

Does the tuned TCN maintain consistent prediction performance across
different source runs and target-behaviour conditions?

### Conclusion

The robustness analysis provides evidence about the stability of the TCN
model rather than relying only on one overall test score. Differences in
error across conditions can help identify difficult cases and guide future
model improvements.

### Important Dataset Limitation

The available dataset does not contain verified cell IDs or explicit
degradation-stage labels. Therefore, `source_file` is treated as a
source/training-run identifier, and the condition groups are treated as
target-behaviour conditions rather than true physical degradation stages.

The experiment predicts `gen_total` from GAN training-log features. It is
therefore an adaptation of the original SoH/RUL-style robustness experiment
and should not be interpreted as actual battery SoH/RUL analysis.